In [42]:
import sqlite3

In [43]:
DB="prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
conn.commit()

In [44]:
def get_ticket_prices(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city=?', (city.lower(),))
        result = cursor.fetchone()
        if result:
            return f'Ticket price to {city} is ${result[0]:.2f}'
        else:
            return f"No Data Available for {city}"

In [45]:
get_ticket_prices("New York")

'Ticket price to New York is $199.99'

In [46]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?)', (city.lower(), price))
        conn.commit()

In [48]:
get_ticket_price("New York")

'Ticket price to New York is $199.99'

In [ ]:
ticket_price ={"london": 150.00, "paris": 120.00, "tokyo": 200.00}
for city, price in ticket_price.items():
    set_ticket_price(city, price)

In [49]:
get_ticket_prices("london")

'Ticket price to london is $150.00'

In [50]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [51]:
groq_api_key = os.getenv("GROQ_API_KEY")
groq_base_url = os.getenv("GROQ_BASE_URL")

if not groq_api_key or not groq_base_url:
    print("GROQ keys not found")
else:
    print("GROQ keys loaded successfully")

GROQ keys loaded successfully


In [52]:
from openai import OpenAI
llm=OpenAI(api_key=groq_api_key, base_url=groq_base_url)


In [60]:
import json
price_function={
    "name":"get_ticket_prices",
    "description":"Get the price of a return ticket to the destination city",
    "parameters":{
        "type":"object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description":"The city that the customer wants to travel to"
            },
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [61]:
tools=[{"type":"function", "function":price_function}]

In [62]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_prices',
   'description': 'Get the price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [63]:
system_message = """
You are a helpful assistant that provides ticket prices for various cities.
You have access to a database of ticket prices for different cities.
When a user asks for the ticket price to a city, you will query the database and return the price.
If the city is not in the database, you will return a message saying that no data is available for that city.
"""

In [64]:
def chat(message, history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]

    messages=[{"role":"system", "content":system_message}]+history+[{"role":"user", "content":message}]


    response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    print(response)
    
    if response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_call(message)
        messages.append(message)
        messages.append(response)

        response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
        )
    return response.choices[0].message.content


In [65]:
def handle_tool_call(message):
    tool_call=message.tool_calls[0]
    if tool_call.function.name=="get_ticket_prices":
        arguments=json.loads(tool_call.function.arguments)
        city=arguments.get('destination_city')
        price_details=get_ticket_prices(city)
        response={
            "role":"tool",
            "content":price_details,
            "tool_call_id":tool_call.id
        }
    return response

In [66]:
import gradio as gr
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://d17ebd4808cbc00091.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ChatCompletion(id='chatcmpl-c08e98db-0fa5-466e-b825-80d40ad07569', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today? If you’d like to know the price of a return ticket to a specific city, just let me know the destination.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to respond. User just says "hi". Probably greet and ask what city they want ticket price for.'))], created=1776141083, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_d81b3304b3', usage=CompletionUsage(completion_tokens=66, prompt_tokens=219, total_tokens=285, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=24, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.0615746, prompt_time=0.05229965, completion_time=0.14144519